In [1]:
# Cell 1 — Imports
from pathlib import Path
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_config, load_model
from gcamp_analysis.experiments.tree import ExperimentTreeBuilder, is_video_dir, print_tree
from gcamp_analysis.video_runner import VideoPipelineRunner
from gcamp_analysis.experiments.processor import ExperimentProcessor
from gcamp_analysis.experiments.io import save_comparisons, save_treatment_comparisons

In [2]:
config_path = PROJECT_ROOT / "config" / "notebook_config.yaml"
config = load_config(config_path)

print(f"Config: {config_path}")

Config: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\notebook_config.yaml


In [3]:
roi_model, roi_cfg = load_model(config["models"], which="roi")
spike_model, spike_cfg = load_model(config["models"], which="spike")

models = {
    "roi": roi_model,
    "roi_config": roi_cfg,
    "spike": spike_model,
    "spike_config": spike_cfg,
}

runner = VideoPipelineRunner.build(config, models)

print(f"ROI model:   {type(roi_model).__name__}")
print(f"Spike model: {type(spike_model).__name__}")

ROI model:   RandomForestClassifier
Spike model: LogisticRegression


In [4]:
EXPERIMENT_ROOT = Path(r"G:\Calcium Imaging\GCaMP6s_EX328 - Copy")  # TODO: change per experiment
assert EXPERIMENT_ROOT.exists(), f"Experiment root not found: {EXPERIMENT_ROOT}"

builder = ExperimentTreeBuilder(is_video_dir=is_video_dir)
tree = builder.build(EXPERIMENT_ROOT)
print_tree(tree)

└── GCaMP6s_EX328 - Copy
    ├── 1uM_CoCl
    │   ├── 2-1
    │   ├── 2-1_1hr-Recovery
    │   ├── 2-1_CoCl_2m
    │   ├── 2-2
    │   ├── 2-2_1hr-Recovery
    │   ├── 2-2_CoCl_4m
    │   ├── 2-3
    │   ├── 2-3_1hr-Recovery
    │   ├── 2-3_CoCl_6m
    │   ├── 2-4
    │   ├── 2-4_1hr-Recovery
    │   ├── 2-4_CoCl_9m
    │   └── metrics
    ├── 1uM_GABA
    │   ├── Week 1
    │   │   ├── 1-1
    │   │   ├── 1-1_GABA_8m
    │   │   ├── 1-2
    │   │   ├── 1-2_GABA_12m
    │   │   ├── 1-3
    │   │   ├── 1-3_GABA_15m
    │   │   ├── 1-4
    │   │   ├── 1-4_GABA_3m
    │   │   └── metrics
    │   ├── Week 2
    │   │   ├── 2-1
    │   │   ├── 2-1_1hr-Recovery
    │   │   ├── 2-1_GABA_1m
    │   │   ├── 2-2
    │   │   ├── 2-2_1hr-Recovery
    │   │   ├── 2-2_GABA_4m
    │   │   ├── 2-3
    │   │   ├── 2-3_1hr-Recovery
    │   │   ├── 2-3_GABA_6m
    │   │   ├── 2-4
    │   │   ├── 2-4_1hr-Recovery
    │   │   ├── 2-4_GABA_10m
    │   │   └── metrics
    │   └── metrics
    ├── 1uM_Glut
   

In [5]:
processor = ExperimentProcessor(
    runner=runner,
    output_root=EXPERIMENT_ROOT,
)
processor.process_tree(tree, verbose=True)


 Processing: 2-1
  Concatenated mode: 3 section(s) (baseline, recovery_1, treatment_1)
  Traces: 181 ROIs, 2700 frames @ 15.0 Hz
  ROI filter: 62/181 kept (34.3%)
  Spikes: 1334/4856 kept | neurons 62 -> 62
  Grouping (combined): | combined=14
  Section comparison (combined, recovery_1): 14 groups | mean delta corr=-0.736 | 0 surviving sub-groups
  Section comparison (combined, treatment_1): 14 groups | mean delta corr=-0.709 | 0 surviving sub-groups

 Processing: 2-2
  Concatenated mode: 3 section(s) (baseline, recovery_1, treatment_1)
  Traces: 298 ROIs, 2700 frames @ 15.0 Hz
  ROI filter: 128/298 kept (43.0%)
  Spikes: 1846/9671 kept | neurons 128 -> 128
  Grouping (combined): | combined=20
  Section comparison (combined, recovery_1): 20 groups | mean delta corr=-0.550 | 1 surviving sub-groups
  Section comparison (combined, treatment_1): 20 groups | mean delta corr=-0.565 | 2 surviving sub-groups

 Processing: 2-3
  Concatenated mode: 3 section(s) (baseline, recovery_1, treatment_

In [6]:
 
sibling_tables = processor.compare_siblings(tree)

for node_path, df in sibling_tables.items():
    if len(df) >= 2:
        print(f"\nNode: {node_path}")
        print(df.to_string(index=False))


Node: G:\Calcium Imaging\GCaMP6s_EX328 - Copy
   child  n_videos  n_neurons  n_groups_combined  mean_group_size_combined  median_group_size_combined  mean_group_corr_combined  mean_spikes_per_group_combined  frac_grouped  frac_ungrouped  decay_tau_seconds_mean_unweighted  half_max_width_seconds_mean_unweighted  rise_slope_hz_mean_unweighted  decay_tau_seconds_mean_weighted  half_max_width_seconds_mean_weighted  rise_slope_hz_mean_weighted  decay_tau_seconds_mean_grouped  half_max_width_seconds_mean_grouped  rise_slope_hz_mean_grouped  decay_tau_seconds_mean_ungrouped  half_max_width_seconds_mean_ungrouped  rise_slope_hz_mean_ungrouped  spike_frequency_mean_unweighted  spike_frequency_mean_weighted  spike_frequency_mean_grouped  spike_frequency_mean_ungrouped  decay_tau_seconds_var_unweighted  decay_tau_seconds_within_unweighted  decay_tau_seconds_between_unweighted  half_max_width_seconds_var_unweighted  half_max_width_seconds_within_unweighted  half_max_width_seconds_between_unweight

In [7]:
save_comparisons(
    root=tree,
    sibling_tables=sibling_tables,
    output_subdir="metrics",
    filename="sibling_comparisons.xlsx",
)

save_treatment_comparisons(tree)